# DNAR flow solver demo

Ноутбук показывает, как запустить DNAR-inspired solver для задачи потоков на JSON-датасете из репозитория. Solver встраивается в контракт `VolumeDataset -> VolumeSolver.solve_checked() -> evaluate()`: DNAR-слой кодирует задачи/агентов/совместимые пары в дискретные состояния, делает несколько processor steps и передает ранжированный датасет существующему построителю маршрутов и repair.

In [ ]:
from pathlib import Path
import sys

REPO = Path.cwd().resolve()
if (REPO / 'Optimization-of-flows').exists():
    FLOW_ROOT = REPO / 'Optimization-of-flows'
else:
    FLOW_ROOT = REPO.parent if REPO.name == 'notebooks' else REPO

sys.path.insert(0, str(FLOW_ROOT / 'src' / 'flowopt'))
FLOW_ROOT

In [ ]:
from volume_core import VolumeDataset, DnarFlowConfig, DnarFlowVolumeSolver

dataset_path = FLOW_ROOT / 'storage' / 'syntetic_data_gap_vrp_solver' / 'data' / 'dataset_sandbox_type2.json'
dataset = VolumeDataset.from_json(dataset_path)
print(f'tasks={len(dataset.tasks)}, agents={len(dataset.agents)}, nodes={len(dataset.nodes)}')

In [ ]:
solver = DnarFlowVolumeSolver(DnarFlowConfig(
    processor_steps=4,
    hidden_size=16,
    max_runtime_sec=30.0,
    verbose=True,
))
solution = solver.solve_checked(dataset)
evaluation = dataset.evaluate(solution)
evaluation.as_dict()

In [ ]:
print('Algorithm:', solution.algorithm)
print('Trips:', len(solution.trips))
print('Unassigned:', len(solution.unassigned_task_ids), solution.unassigned_task_ids[:10])
print('First DNAR logs:')
for line in solution.solver_logs[:8]:
    print(' ', line)